In [1]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [2]:
# load dataset
df = pd.read_csv("news.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df["label"].value_counts())

Dataset shape: (6335, 4)
   Unnamed: 0                                              title  \
0        8476                       You Can Smell Hillary’s Fear   
1       10294  Watch The Exact Moment Paul Ryan Committed Pol...   
2        3608        Kerry to go to Paris in gesture of sympathy   
3       10142  Bernie supporters on Twitter erupt in anger ag...   
4         875   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  
label
REAL    3171
FAKE    3164
Name: count, dtype: int64


In [3]:
# data preprocessing
df = df.dropna(subset=["label"]).copy()

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

df["content"] = (
    df["title"].astype(str) + " " +
    df["text"].astype(str)
).str.strip()

df = df[df["content"] != ""]
df = df.drop_duplicates(subset=["content"])

X = df["content"]
y = df["label"].str.strip().str.upper()

print("Total articles:", len(df))
print("Labels:", y.unique())
print("Missing values:", X.isna().sum())

Total articles: 6306
Labels: <ArrowStringArray>
['FAKE', 'REAL']
Length: 2, dtype: str
Missing values: 0


In [4]:
#split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 5044
Testing samples: 1262


In [5]:
#build the TF-IDF model
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Training feature shape:", X_train_tfidf.shape)
print("Testing feature shape:", X_test_tfidf.shape)

Training feature shape: (5044, 50000)
Testing feature shape: (1262, 50000)


In [6]:
#training
tfidf_model = LogisticRegression(max_iter=1000)

tfidf_model.fit(X_train_tfidf, y_train)

tfidf_predictions = tfidf_model.predict(X_test_tfidf)

print("TF-IDF Accuracy:",
      f"{accuracy_score(y_test, tfidf_predictions):.2%}")

print(classification_report(y_test, tfidf_predictions))

TF-IDF Accuracy: 92.63%
              precision    recall  f1-score   support

        FAKE       0.92      0.94      0.93       631
        REAL       0.94      0.91      0.93       631

    accuracy                           0.93      1262
   macro avg       0.93      0.93      0.93      1262
weighted avg       0.93      0.93      0.93      1262



In [7]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer

from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)

df = pd.read_csv("news.csv")

df = df.dropna(subset=["label"]).copy()
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["label"] = df["label"].str.strip().str.upper()

df["content"] = (
    df["title"].astype(str) + " " + df["text"].astype(str)
).str.strip()

df = df[df["content"] != ""]
df = df[df["label"].isin(["REAL", "FAKE"])]
df = df.drop_duplicates(subset=["content"])

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Total records:", len(df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Total records: 6306
Training records: 5044
Testing records: 1262


In [8]:
# Create TF-IDF features
tfidf_vectorizer = TfidfVectorizer(
    max_features=30000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (5044, 30000)
Testing shape: (1262, 30000)


In [9]:
# Multinomial Naive Bayes
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

pred_mnb_tfidf = mnb_tfidf.predict(X_test_tfidf)

print("TF-IDF + Multinomial Naive Bayes")
print("Accuracy:", accuracy_score(y_test, pred_mnb_tfidf))
print(classification_report(y_test, pred_mnb_tfidf))

TF-IDF + Multinomial Naive Bayes
Accuracy: 0.901743264659271
              precision    recall  f1-score   support

        FAKE       0.91      0.89      0.90       631
        REAL       0.89      0.91      0.90       631

    accuracy                           0.90      1262
   macro avg       0.90      0.90      0.90      1262
weighted avg       0.90      0.90      0.90      1262



In [10]:
# Bernoulli Naive Bayes
bnb_tfidf = BernoulliNB()
bnb_tfidf.fit(X_train_tfidf, y_train)

pred_bnb_tfidf = bnb_tfidf.predict(X_test_tfidf)

print("TF-IDF + Bernoulli Naive Bayes")
print("Accuracy:", accuracy_score(y_test, pred_bnb_tfidf))
print(classification_report(y_test, pred_bnb_tfidf))

TF-IDF + Bernoulli Naive Bayes
Accuracy: 0.8343898573692552
              precision    recall  f1-score   support

        FAKE       0.78      0.94      0.85       631
        REAL       0.92      0.73      0.82       631

    accuracy                           0.83      1262
   macro avg       0.85      0.83      0.83      1262
weighted avg       0.85      0.83      0.83      1262



In [11]:
# Linear SVM
svm_tfidf = LinearSVC()
svm_tfidf.fit(X_train_tfidf, y_train)

pred_svm_tfidf = svm_tfidf.predict(X_test_tfidf)

print("TF-IDF + Linear SVM")
print("Accuracy:", accuracy_score(y_test, pred_svm_tfidf))
print(classification_report(y_test, pred_svm_tfidf))

TF-IDF + Linear SVM
Accuracy: 0.9326465927099842
              precision    recall  f1-score   support

        FAKE       0.93      0.94      0.93       631
        REAL       0.94      0.93      0.93       631

    accuracy                           0.93      1262
   macro avg       0.93      0.93      0.93      1262
weighted avg       0.93      0.93      0.93      1262



In [12]:
# Random Forest
rf_tfidf = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,
    random_state=42,
    n_jobs=-1
)

rf_tfidf.fit(X_train_tfidf, y_train)
pred_rf_tfidf = rf_tfidf.predict(X_test_tfidf)

print("TF-IDF + Random Forest")
print("Accuracy:", accuracy_score(y_test, pred_rf_tfidf))
print(classification_report(y_test, pred_rf_tfidf))

TF-IDF + Random Forest
Accuracy: 0.8890649762282092
              precision    recall  f1-score   support

        FAKE       0.87      0.91      0.89       631
        REAL       0.91      0.87      0.89       631

    accuracy                           0.89      1262
   macro avg       0.89      0.89      0.89      1262
weighted avg       0.89      0.89      0.89      1262



In [13]:
# Decision Tree
dt_tfidf = DecisionTreeClassifier(
    max_depth=30,
    random_state=42
)

dt_tfidf.fit(X_train_tfidf, y_train)
pred_dt_tfidf = dt_tfidf.predict(X_test_tfidf)

print("TF-IDF + Decision Tree")
print("Accuracy:", accuracy_score(y_test, pred_dt_tfidf))
print(classification_report(y_test, pred_dt_tfidf))

TF-IDF + Decision Tree
Accuracy: 0.8026941362916006
              precision    recall  f1-score   support

        FAKE       0.80      0.81      0.80       631
        REAL       0.81      0.79      0.80       631

    accuracy                           0.80      1262
   macro avg       0.80      0.80      0.80      1262
weighted avg       0.80      0.80      0.80      1262



In [14]:
# K-Nearest Neighbors (KNN)
knn_tfidf = KNeighborsClassifier(
    n_neighbors=5,
    algorithm="brute",
    metric="cosine"
)

knn_tfidf.fit(X_train_tfidf, y_train)
pred_knn_tfidf = knn_tfidf.predict(X_test_tfidf)

print("TF-IDF + KNN")
print("Accuracy:", accuracy_score(y_test, pred_knn_tfidf))
print(classification_report(y_test, pred_knn_tfidf))

TF-IDF + KNN
Accuracy: 0.9001584786053882
              precision    recall  f1-score   support

        FAKE       0.94      0.85      0.90       631
        REAL       0.87      0.95      0.90       631

    accuracy                           0.90      1262
   macro avg       0.90      0.90      0.90      1262
weighted avg       0.90      0.90      0.90      1262



In [15]:
# Linear Regression
y_train_num = (y_train == "REAL").astype(int)
y_test_num = (y_test == "REAL").astype(int)

lr_tfidf = LinearRegression()
lr_tfidf.fit(X_train_tfidf, y_train_num)

pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

print("TF-IDF + Linear Regression")
print("MAE:", mean_absolute_error(y_test_num, pred_lr_tfidf))
print("RMSE:", np.sqrt(mean_squared_error(y_test_num, pred_lr_tfidf)))
print("R2 Score:", r2_score(y_test_num, pred_lr_tfidf))

TF-IDF + Linear Regression
MAE: 0.2092759066463458
RMSE: 0.2721007038558713
R2 Score: 0.7038448278445577


In [20]:
comparison_data = []

predictions = {
    "Multinomial NB - TF-IDF": pred_mnb_tfidf,
    
    "Bernoulli NB - TF-IDF": pred_bnb_tfidf,
    
    "Linear SVM - TF-IDF": pred_svm_tfidf,
    
    "Random Forest - TF-IDF": pred_rf_tfidf,

    "Decision Tree - TF-IDF": pred_dt_tfidf,
    
    "KNN - TF-IDF": pred_knn_tfidf,
    
}

for name, pred in predictions.items():
    comparison_data.append({
        "Model": name,
        "Accuracy (%)": round(accuracy_score(y_test, pred) * 100, 2),
        "Precision (%)": round(precision_score(
            y_test, pred, pos_label="FAKE", zero_division=0
        ) * 100, 2),
        "Recall (%)": round(recall_score(
            y_test, pred, pos_label="FAKE", zero_division=0
        ) * 100, 2),
        "F1 Score (%)": round(f1_score(
            y_test, pred, pos_label="FAKE", zero_division=0
        ) * 100, 2)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values(
    "F1 Score (%)", ascending=False
).reset_index(drop=True)

display(comparison_df)
comparison_df.to_csv("tfid_comparison.csv", index=False)

,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Linear SVM - TF-IDF,93.26,92.66,93.98,93.31
1,Multinomial NB - TF-IDF,90.17,90.95,89.22,90.08
2,KNN - TF-IDF,90.02,94.07,85.42,89.53
3,Random Forest - TF-IDF,88.91,87.25,91.13,89.15
4,Bernoulli NB - TF-IDF,83.44,77.69,93.82,85.00
5,Decision Tree - TF-IDF,80.27,79.75,81.14,80.44
